## 1 - Importações

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2 - Carregamento dos dados

In [2]:
df = pd.read_excel(
    "../data/raw/base-seguros.xlsx",
    sheet_name="Base"
)

df.head()

,Flag_Renovou,Idade,Perfil_Risco,Diferenca_Perfil,Genero,Profissao,Tempo_Apolice,Uso_Veiculo,Qte_Apolices,Premio_Final,Premio_Qte_Parc,Premio_Pago_Ult,Premio_Mercado,Premio_Orig,Veic_Idade,Veic_Idade_Compra,Veic_Garagem,Veic_Potencia,Veic_Regiao
0,0,38,stable,only partner,Male,normal,1,private or freelance work,1,232.46,4 per year,232.47,221.56,243.59,9,8,private garage,225 kW,Reg7
1,1,35,stable,same,Male,normal,1,private or freelance work,1,208.53,4 per year,208.54,247.56,208.54,15,7,private garage,100 kW,Reg4
2,1,29,stable,same,Male,normal,0,private or freelance work,1,277.34,1 per year,277.35,293.32,277.35,14,6,underground garage,100 kW,Reg7
3,0,33,down,same,Female,medical,2,private or freelance work,1,239.51,4 per year,244.40,310.91,219.95,17,10,street,75 kW,Reg5
4,0,50,stable,same,Male,normal,8,unknown,1,554.54,4 per year,554.55,365.46,519.50,16,8,street,75 kW,Reg14


## 3 - Dimensões

In [3]:
df.shape

(23060, 19)

##### Obs.: A base contém aproximadamente 23 mil registros de clientes e 19 variáveis relacionadas ao perfil do cliente, características da apólice e do veículo, além da decisão de renovação.

## 4 - Estrutura das variáveis

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23060 entries, 0 to 23059
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Flag_Renovou       23060 non-null  int64  
 1   Idade              23060 non-null  int64  
 2   Perfil_Risco       23060 non-null  str    
 3   Diferenca_Perfil   23060 non-null  str    
 4   Genero             23060 non-null  str    
 5   Profissao          23060 non-null  str    
 6   Tempo_Apolice      23060 non-null  int64  
 7   Uso_Veiculo        23060 non-null  str    
 8   Qte_Apolices       23060 non-null  int64  
 9   Premio_Final       23060 non-null  float64
 10  Premio_Qte_Parc    23060 non-null  str    
 11  Premio_Pago_Ult    23060 non-null  float64
 12  Premio_Mercado     23060 non-null  float64
 13  Premio_Orig        23060 non-null  float64
 14  Veic_Idade         23060 non-null  int64  
 15  Veic_Idade_Compra  23060 non-null  int64  
 16  Veic_Garagem       23060 non-null

In [5]:
pd.DataFrame({
    "variavel": df.columns,
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum().values,
    "n_unicos": df.nunique().values
})

,variavel,tipo,nulos,n_unicos
Flag_Renovou,Flag_Renovou,int64,0,2
Idade,Idade,int64,0,67
Perfil_Risco,Perfil_Risco,str,0,3
Diferenca_Perfil,Diferenca_Perfil,str,0,7
Genero,Genero,str,0,2
Profissao,Profissao,str,0,2
Tempo_Apolice,Tempo_Apolice,int64,0,18
Uso_Veiculo,Uso_Veiculo,str,0,3
Qte_Apolices,Qte_Apolices,int64,0,15
Premio_Final,Premio_Final,float64,0,14669


## 5 - Valores ausentes

In [10]:
nulos = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

nulos[nulos > 0]
print("Valores ausentes:", len(nulos[nulos > 0]))

Valores ausentes: 0


## 6 - Duplicidades

In [14]:
duplicados = df.duplicated().sum()
print("Duplicatas:", duplicados)

Duplicatas: 0


In [15]:
duplicados = df.duplicated().mean()
print("Duplicatas:", df.duplicated().mean())

Duplicatas: 0.0


## 7 - Target: Flag_Renovou

In [17]:
df["Flag_Renovou"].value_counts()

Flag_Renovou
0    20106
1     2954
Name: count, dtype: int64

In [18]:
taxa_renovacao = (
    df["Flag_Renovou"]
      .value_counts(normalize=True)
      .mul(100)
      .rename("percentual")
      .reset_index()
)

taxa_renovacao

,Flag_Renovou,percentual
0,0,87.189939
1,1,12.810061


#### Obs.: Temos um forte desbalanceamento da variável alvo

## 8 - KPI (inicial)

In [19]:
taxa_renovacao = df["Flag_Renovou"].mean()

print(f"Taxa de renovação: {taxa_renovacao:.2%}")
print(f"Taxa de não renovação: {1 - taxa_renovacao:.2%}")

Taxa de renovação: 12.81%
Taxa de não renovação: 87.19%


##### Obs.: Aproximadamente 1 em cada X clientes não estão renovando suas apólices.

## 9 - Estatísticas das variáveis numéricas

In [20]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Flag_Renovou,23060.0,0.128101,0.334209,0.00,0.0000,0.000,0.0000,1.00
Idade,23060.0,43.045490,12.352291,19.00,35.0000,41.000,49.0000,85.00
Tempo_Apolice,23060.0,2.443452,3.100771,0.00,0.0000,1.000,4.0000,17.00
Qte_Apolices,23060.0,1.305637,0.788647,1.00,1.0000,1.000,1.0000,15.00
Premio_Final,23060.0,374.123791,212.899174,46.55,232.8375,312.250,448.3700,2948.05
Premio_Pago_Ult,23060.0,380.508774,227.937859,46.56,232.6300,311.005,449.6025,3362.07
Premio_Mercado,23060.0,373.528631,201.915809,50.11,245.1500,316.830,434.4525,2416.84
Premio_Orig,23060.0,355.882315,197.138010,45.55,227.1000,301.445,423.5625,2716.08
Veic_Idade,23060.0,13.060624,3.590088,0.00,11.0000,13.000,16.0000,18.00
Veic_Idade_Compra,23060.0,7.680876,4.960701,0.00,4.0000,8.000,11.0000,18.00


## 10 - Variáveis categóricas

In [21]:
variaveis_categoricas = df.select_dtypes(
    include=["object", "category", "str"]
).columns

for col in variaveis_categoricas:
    print(f"\n{col}")
    print(df[col].nunique())
    print(df[col].value_counts().head(10))


Perfil_Risco
3
Perfil_Risco
stable    12036
down      10155
up          869
Name: count, dtype: int64

Diferenca_Perfil
7
Diferenca_Perfil
same                11155
only partner         8128
young drivers        1955
all drivers > 24     1728
learner 17             42
commercial             40
unknown                12
Name: count, dtype: int64

Genero
2
Genero
Male      14721
Female     8339
Name: count, dtype: int64

Profissao
2
Profissao
normal     13578
medical     9482
Name: count, dtype: int64

Uso_Veiculo
3
Uso_Veiculo
private or freelance work    19567
unknown                       3483
commercial                      10
Name: count, dtype: int64

Premio_Qte_Parc
4
Premio_Qte_Parc
1 per year     11680
4 per year      6114
2 per year      3090
12 per year     2176
Name: count, dtype: int64

Veic_Garagem
8
Veic_Garagem
private garage        8863
street                5468
other                 2243
parking deck          2243
unknown               1575
carport               1413


In [22]:
variaveis_categoricas = [
    "Perfil_Risco",
    "Diferenca_Perfil",
    "Genero",
    "Profissao",
    "Uso_Veiculo",
    "Premio_Qte_Parc",
    "Veic_Garagem",
    "Veic_Potencia",
    "Veic_Regiao"
]

for col in variaveis_categoricas:

    resultado = (
        df.groupby(col)
          .agg(
              clientes=("Flag_Renovou", "count"),
              taxa_renovacao=("Flag_Renovou", "mean")
          )
          .sort_values("taxa_renovacao")
    )

    resultado["taxa_renovacao"] *= 100

    print(f"\n{'='*70}")
    print(col)
    display(resultado)


Perfil_Risco


,clientes,taxa_renovacao
Perfil_Risco,,
up,869,9.781358
stable,12036,10.227650
down,10155,16.129985



Diferenca_Perfil


,clientes,taxa_renovacao
Diferenca_Perfil,,
commercial,40,5.000000
all drivers > 24,1728,11.863426
same,11155,12.254594
only partner,8128,12.832185
unknown,12,16.666667
young drivers,1955,16.726343
learner 17,42,19.047619



Genero


...

,clientes,taxa_renovacao
Veic_Garagem,,
private estate,199,8.040201
street,5468,12.362838
private garage,8863,12.490127
parking deck,2243,12.661614
other,2243,12.750780
carport,1413,13.658882
unknown,1575,13.777778
underground garage,1056,16.571970



Veic_Potencia


,clientes,taxa_renovacao
Veic_Potencia,,
300 kW,2,0.000000
275 kW,4,0.000000
200 kW,32,6.250000
225 kW,77,10.389610
125-300 kW,1720,11.918605
150 kW,580,12.068966
175 kW,206,12.135922
25-50 kW,4968,12.278583
250 kW,16,12.500000



Veic_Regiao


,clientes,taxa_renovacao
Veic_Regiao,,
Reg2,556,7.374101
Reg5,2391,10.037641
Reg1,618,10.517799
Reg7,3156,11.850444
Reg6,1119,12.064343
Reg3,1827,12.205802
Reg4,4325,12.670520
Reg8,3074,12.979831
Reg10,1057,13.150426


## 11 Variáveis Numéricas

In [18]:
# Divindo clientes em faixas de idade
df["Faixa_Idade"] = pd.cut(
    df["Idade"],
    bins=[18, 30, 40, 50, 60, 100],
    labels=[
        "19-30",
        "31-40",
        "41-50",
        "51-60",
        "61+"
    ]
)

idade_renovacao = (
    df.groupby("Faixa_Idade", observed=True)
      .agg(
          clientes=("Flag_Renovou", "count"),
          taxa_renovacao=("Flag_Renovou", "mean")
      )
)

idade_renovacao["taxa_renovacao"] *= 100

idade_renovacao

,clientes,taxa_renovacao
Faixa_Idade,,
19-30,2974,17.283120
31-40,8198,13.381313
41-50,6701,12.177287
51-60,2841,11.193242
61+,2346,8.908781


> **Insight preliminar:** Clientes mais jovens apresentam taxas de renovação superiores às observadas entre clientes mais velhos, com uma diferença de aproximadamente **8,4 pontos percentuais** entre os grupos extremos.

In [19]:
# Dividindo por faixa de tempo de apólice
df["Faixa_Tempo_Apolice"] = pd.cut(
    df["Tempo_Apolice"],
    bins=[-1, 1, 3, 5, 10, 20],
    labels=[
        "0-1",
        "2-3",
        "4-5",
        "6-10",
        "11+"
    ]
)

tempo_renovacao = (
    df.groupby("Faixa_Tempo_Apolice", observed=True)
      .agg(
          clientes=("Flag_Renovou", "count"),
          taxa_renovacao=("Flag_Renovou", "mean")
      )
)

tempo_renovacao["taxa_renovacao"] *= 100

tempo_renovacao

,clientes,taxa_renovacao
Faixa_Tempo_Apolice,,
0-1,13189,14.148154
2-3,3847,13.491032
4-5,1683,12.953060
6-10,3942,8.219178
11+,399,6.766917


> **Insight preliminar:** Quanto maior o tempo de apólice, **menor a taxa de renovação**.

In [20]:
# Dividindo clientes em faixa de prêmio
df["Faixa_Premio"] = pd.qcut(
    df["Premio_Final"],
    q=4,
    duplicates="drop"
)

premio_renovacao = (
    df.groupby("Faixa_Premio", observed=True)
      .agg(
          clientes=("Flag_Renovou", "count"),
          taxa_renovacao=("Flag_Renovou", "mean")
      )
)

premio_renovacao["taxa_renovacao"] *= 100

premio_renovacao

,clientes,taxa_renovacao
Faixa_Premio,,
"(46.549, 232.838]",5765,10.494363
"(232.838, 312.25]",5770,12.582322
"(312.25, 448.37]",5760,13.437500
"(448.37, 2948.05]",5765,14.726800


> **Insight preliminar:**  Clientes com maior prêmio apresentam maior taxa de renovação.